In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
import numpy as np
import pandas as pd
import os, json, shutil
from datetime import datetime

os.makedirs("../models/als",    exist_ok=True)
os.makedirs("../models/shared", exist_ok=True)

spark = SparkSession.builder \
    .appName("ExportMatrices") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .config("spark.driver.maxResultSize", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} ✓")
print("Dossiers créés ✓")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/15 12:23:52 WARN Utils: Your hostname, MacBook-M4-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.10.137.64 instead (on interface en0)
26/04/15 12:23:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/15 12:23:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.1.1 ✓
Dossiers créés ✓


In [2]:
LARGE = "../data/processed/32m/"

ratings = spark.read.parquet(f"{LARGE}ratings_clean.parquet")
movies  = spark.read.parquet(f"{LARGE}movies_clean.parquet")
links   = spark.read.csv("../data/32m/links.csv", header=True, inferSchema=True)

print(f"Ratings : {ratings.count():,}")
print(f"Movies  : {movies.count():,}")
print(f"Links   : {links.count():,}")

Ratings : 32,000,204
Movies  : 87,585
Links   : 87,585


In [3]:
als = ALS(
    rank=5,
    maxIter=10,
    regParam=0.1,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    seed=42
)

model = als.fit(ratings)
print("ALS model trained ✓")

ALS model trained ✓


In [4]:
user_factors_df = model.userFactors.orderBy("id").toPandas()
item_factors_df = model.itemFactors.orderBy("id").toPandas()

user_ids     = user_factors_df["id"].values.astype(np.int32)
user_vectors = np.array(user_factors_df["features"].tolist(), dtype=np.float32)
item_ids     = item_factors_df["id"].values.astype(np.int32)
item_vectors = np.array(item_factors_df["features"].tolist(), dtype=np.float32)

print(f"User matrix : {user_vectors.shape}")
print(f"Item matrix : {item_vectors.shape}")

np.save("../models/als/user_ids.npy",     user_ids)
np.save("../models/als/user_vectors.npy", user_vectors)
np.save("../models/als/item_ids.npy",     item_ids)
np.save("../models/als/item_vectors.npy", item_vectors)

print("Matrices saved ✓")

User matrix : (200948, 5)
Item matrix : (84432, 5)
Matrices saved ✓


In [5]:
metadata = {
    "model":      "ALS",
    "dataset":    "large (32M ratings)",
    "rank":       5,
    "maxIter":    10,
    "regParam":   0.1,
    "rmse":       0.8033,
    "n_users":    int(user_vectors.shape[0]),
    "n_items":    int(item_vectors.shape[0]),
    "trained_at": datetime.now().isoformat()
}

with open("../models/als/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Metadata saved ✓")

Metadata saved ✓


In [6]:
movies_pd = movies.select("movieId", "title", "genres").toPandas()
movies_pd.to_csv("../models/shared/movies.csv", index=False)

links_pd = links.select("movieId", "tmdbId").toPandas()
links_pd = links_pd.dropna(subset=["tmdbId"])
links_pd["tmdbId"] = links_pd["tmdbId"].astype(int)
links_pd.to_csv("../models/shared/links.csv", index=False)

movie_id_to_idx = {int(mid): int(idx) for idx, mid in enumerate(item_ids)}
with open("../models/shared/movie_id_to_idx.json", "w") as f:
    json.dump(movie_id_to_idx, f)

ratings_count = ratings.groupBy("movieId").count().toPandas()
ratings_count.to_csv("../models/shared/ratings_count.csv", index=False)

spark.stop()

print(f"Movies        : {len(movies_pd):,}")
print(f"Links         : {len(links_pd):,}")
print(f"Movie index   : {len(movie_id_to_idx):,}")
print(f"Ratings count : {len(ratings_count):,}")
print("Export complet ✓")

Movies        : 87,585
Links         : 87,461
Movie index   : 84,432
Ratings count : 84,432
Export complet ✓
